In [58]:
import numpy as np 
import matplotlib.pyplot as plt
import pandas as pd
from pathlib import Path
from scipy import stats
from scipy.stats import rankdata
from scipy.optimize import linear_sum_assignment
from scipy.stats import spearmanr
from numpy import random

In [59]:
#number of trials of each type. 
# Practice trials have Economy-Knowledge-Politics
# Easy trials have Military-Knowledge-Politics
# Hard trials have Economy-Military-Culture
# 5 objective have all 5 objectives
#
nprac = 0
neasy = 0
nhard = 1
nfive = 0
ntrials = nprac+neasy+nhard+nfive 

#seed to set for each subject
subject_seed = 101

In [60]:
#parameters for the design.
n_items = 60
n_con = 1
objective_range = 20  #
#Right now objective range is the same for objectives and constraints.  
# That means the average card costs $10.  So, capacity should be set to $10*n_selected 
#beta parameters
a = [1.5,1.2,1.3,1.6,1.4,1]
b = [2.1,1.8,1.6,1.9,2.0,1]
a_p = [1.5,1.3,1.6,1]
b_p = [2.1,1.6,1.9,1]
a_e = [1.2,1.3,1.4,1]
b_e = [1.8,1.6,2.0,1]
a_h = [1.5,1.2,1.6,1]
b_h = [2.1,1.8,1.9,1]

#constraint parameters
cost_corr = 0.3

In [61]:
def make_pos_def(corr):
    eigvals, eigvecs = np.linalg.eigh(corr)
    eigvals[eigvals < 1e-8] = 1e-8  
    corr_pd = eigvecs @ np.diag(eigvals) @ eigvecs.T

    # normalize diagonal to 1
    d = np.sqrt(np.diag(corr_pd))
    corr_pd = corr_pd / d[:, None] / d[None, :]
    return corr_pd

In [62]:
def gamma_GC(R, n, shape,scale,rng=None):
    R = make_pos_def(R)
    mean = np.zeros(np.shape(R)[0])
    if rng is None:
        z = np.random.multivariate_normal(mean, R, size=n)
    else:
        z = rng.multivariate_normal(mean, R, size=n)
    u = stats.norm.cdf(z)
    data = np.zeros_like(u)
    for i in range(u.shape[1]):
        data[:, i] = stats.gamma.ppf(u[:, i], a=shape[i], scale=scale[i])
    return data

In [63]:
def beta_GC(R, n, a,b,rng=None):
    R = make_pos_def(R)
    mean = np.zeros(np.shape(R)[0])
    if rng is None:
        z = np.random.multivariate_normal(mean, R, size=n)
    else:
        z = rng.multivariate_normal(mean, R, size=n)
    u = stats.norm.cdf(z)
    data = np.zeros_like(u)
    for i in range(u.shape[1]):
        data[:, i] = stats.beta.ppf(u[:, i], a[i], b[i])
    return data

In [64]:
def cleanupsamples(samples, nobj, precision=1):
    """Clean up samples by rounding and removing duplicates."""
    samples = np.round(samples, precision)
    c, i = np.unique(samples[:, :nobj], axis=0, return_index=True)
    newsamples = samples[i, :]  # note - these have been sorted into increasing magnitude
    if precision == 0:
        newsamples = np.array(newsamples, dtype=np.int32)
    return newsamples

In [65]:
def generate_beta_example_data(r, a, b, max_val,nobj, n_items=100, seed=1124, precision = 0):
    #r = make_pos_def(r)
    item_rng = random.default_rng(seed=seed)
    
    batch = max(5, n_items // 10)
    uniq = set()
    items = []
    while len(items) < n_items:
        new = 1+beta_GC(r, batch, a, b, rng=item_rng)*(max_val-2)
        new = cleanupsamples(new, nobj=nobj, precision=precision)
        for item in new:
            key = tuple(item)
            if key not in uniq:
                uniq.add(key)
                items.append(item)
                if len(items) == n_items:
                    break
    return np.unique(np.array(items), axis=0), r # here np.unique is used for sorting

In [66]:
def rank_matrix(X, method="average"):
    """
    X: (n, d) array-like
    returns: (n, d) matrix of ranks within each column (1..n)
    """
    X = np.asarray(X)
    Y = np.apply_along_axis(rankdata, 0, X, method=method)
    return Y

In [67]:
def sort_B_to_match_A_ranks(A, B, method="average"):
    """
    Reorder rows of B to best match rank patterns of A across columns.

    A, B: (n, d) arrays or DataFrames with same shape (n, 3)
    returns: (B_sorted, perm) where perm are indices into original B
    """
    A_arr = np.asarray(A)
    B_arr = np.asarray(B)
    
    n, d = A_arr.shape
    RA = rank_matrix(A_arr, method=method)
    RB = rank_matrix(B_arr, method=method)
    # cost[i, j] = distance between row i ofhttps://www.cnn.com/2026/02/13/sport/ilia-malinin-winter-olympics-shock A and row j of B in rank space
    diff = RA[:, None, :] - RB[None, :, :]
    
    cost = np.sum((diff ** 2), axis=2)

    row_ind, col_ind = linear_sum_assignment(cost)  # minimizes sum cost[row_ind[k], col_ind[k]]
    # row_ind should be [0..n-1] but keep it general
    perm = col_ind[np.argsort(row_ind)]
    B_sorted = B_arr[perm]

    return B_sorted, perm



In [68]:
def column_spearman(A, B_sorted):
    A = np.asarray(A)
    B_sorted = np.asarray(B_sorted)

    d = A.shape[1]
    corrs = np.zeros(d)

    for j in range(d):
        corrs[j] = spearmanr(A[:, j], B_sorted[:, j]).correlation

    return corrs

In [69]:
def fit_beta_known_bounds(x, L, U, eps=1e-9):
    x = np.asarray(x)
    print(x)
    y = (x - L) / (U - L)
    # avoid exact 0 or 1 (Beta density can be infinite there for some shapes)
    y = np.clip(y, eps, 1 - eps)
    print(y)
    a, b, loc, scale = stats.beta.fit(y, floc=0, fscale=1)   # MLE on [0,1]
    # return parameters for x on [L,U]
    return a,b



In [70]:
def transform_beta_items_to_z(items, scale=100):
    alpha = np.empty(items.shape[1])
    beta = np.empty(items.shape[1])
    items_z = np.empty(items.shape)
    for i in range(items.shape[1]):#get useful variables 
        y = items[:,i]/scale
        a, b, loc, s = stats.beta.fit(y, floc=0, fscale=1)
        alpha[i] = a
        beta[i] = b
        u = stats.beta.cdf(y, a = a, b = b)
        u = np.clip(u, 1e-12, 1-1e-12)
        items_z[:,i] = stats.norm.ppf(u) 
    return alpha, beta,items_z


In [71]:
#load the template file 
city_items = pd.read_csv('civ_deck_60.csv')
city_data = city_items.values[:,1:].astype(float)
#print(city_items.columns)
#built 3 items data frames by dropping items 
practice_items = city_items.drop({'Military','Politics'},axis=1)
practice_data = practice_items.values[:,1:].astype(float)
easy_items = city_items.drop({'Economy','Culture'},axis=1)
easy_data = easy_items.values[:,1:].astype(float)
hard_items = city_items.drop({'Knowledge','Politics'},axis=1)
hard_data = hard_items.values[:,1:].astype(float)

#create output directory if necessary
stimuli_path = Path("stimuli")
stimuli_path.mkdir(parents=True, exist_ok=True)


In [72]:
#estimate correlation matrix from the template
alpha, beta, items_z = transform_beta_items_to_z(city_data)
correlations = np.corrcoef(items_z.T)
n_obj = 5
r = -0.3 * np.ones((n_obj + 1, n_obj + 1)) + (1 + 0.3) * np.eye(n_obj + 1)  #right now not dealing with different r for obj and con
#this is hard coded to allneg case
r[:,n_obj] = cost_corr
r[n_obj, :] = cost_corr
r[n_obj,n_obj] = 1
r[:n_obj, :n_obj] = np.round(correlations,2)
#print(r)

In [73]:
#estimate correlation matrix from the template
alpha, beta, items_z = transform_beta_items_to_z(practice_data)
correlations = np.corrcoef(items_z.T)
n_obj = 3
r_p = -0.3 * np.ones((n_obj + 1, n_obj + 1)) + (1 + 0.3) * np.eye(n_obj + 1)  #right now not dealing with different r for obj and con
#this is hard coded to allneg case
r_p[:,n_obj] = 0.3
r_p[n_obj, :] = 0.3
r_p[n_obj,n_obj] = 1
r_p[:n_obj, :n_obj] = np.round(correlations,2)
#print(r_p)
#estimate correlation matrix from the template
alpha, beta, items_z = transform_beta_items_to_z(easy_data)
correlations = np.corrcoef(items_z.T)
n_obj = 3
r_e = -0.3 * np.ones((n_obj + 1, n_obj + 1)) + (1 + 0.3) * np.eye(n_obj + 1)  #right now not dealing with different r for obj and con
#this is hard coded to allneg case
r_e[:,n_obj] = 0.3
r_e[n_obj, :] = 0.3
r_e[n_obj,n_obj] = 1
r_e[:n_obj, :n_obj] = np.round(correlations,2)
#print(r_e)
#estimate correlation matrix from the template
alpha, beta, items_z = transform_beta_items_to_z(hard_data)
correlations = np.corrcoef(items_z.T)
n_obj = 3
r_h = -0.3 * np.ones((n_obj + 1, n_obj + 1)) + (1 + 0.3) * np.eye(n_obj + 1)  #right now not dealing with different r for obj and con
#this is hard coded to allneg case
r_h[:,n_obj] = 0.3
r_h[n_obj, :] = 0.3
r_h[n_obj,n_obj] = 1
r_h[:n_obj, :n_obj] = np.round(correlations,2)
#print(r_h)

In [ ]:
itrial = 0 
n_obj = 3

corrs_p = np.zeros((nprac,n_obj))
for j in range(nprac):
    items_seed = subject_seed + j
    items, rpos = generate_beta_example_data(r_p, a_p, b_p, max_val = objective_range,nobj = n_obj, n_items=n_items, seed=items_seed)
    itemsdf = pd.DataFrame(items, columns= list(practice_items.columns[1:])+['Cost'])
    #this creates a mapping between the template and the trial data 
    items_sorted_practice_data, perm = sort_B_to_match_A_ranks(practice_data, items[:, :n_obj])
    items_sorted_df = itemsdf.iloc[perm].reset_index(drop=True)
    items_sorted_df.insert(0, practice_items.columns[0], practice_items.iloc[:,0].values)
    #sns.histplot(itemsdf,multiple = 'dodge',kde=True,bins = np.arange(0.5, objective_range + 0.5, 1))
    for k in range(n_obj):
        corrs_p[j,k] = spearmanr(items_sorted_practice_data[:, k], practice_data[:, k]).correlation
    items_sorted_df.to_csv(f'stimuli/civ_items_{items_seed}_trial_{itrial}.csv', index=False)
    itrial += 1

n_obj = 3

corrs_e = np.zeros((neasy,n_obj))
for j in range(neasy):
    items_seed = subject_seed + nprac + j
    items, rpos = generate_beta_example_data(r_e, a_e, b_e, max_val = objective_range,nobj = n_obj, n_items=n_items, seed=items_seed)
    itemsdf = pd.DataFrame(items, columns= list(easy_items.columns[1:])+['Cost'])
    #this creates a mapping between the template and the trial data 
    items_sorted_easy_data, perm = sort_B_to_match_A_ranks(easy_data, items[:, :n_obj])
    items_sorted_df = itemsdf.iloc[perm].reset_index(drop=True)
    items_sorted_df.insert(0, easy_items.columns[0], easy_items.iloc[:,0].values)
    #sns.histplot(itemsdf,multiple = 'dodge',kde=True,bins = np.arange(0.5, objective_range + 0.5, 1))
    for k in range(n_obj):
        corrs_e[j,k] = spearmanr(items_sorted_easy_data[:, k], easy_data[:, k]).correlation
    items_sorted_df.to_csv(f'stimuli/civ_items_{items_seed}_trial_{itrial}.csv', index=False)
    itrial += 1

n_obj = 3

corrs_h = np.zeros((nhard,n_obj))
for j in range(nhard):
    items_seed = subject_seed + nprac + neasy + j
    items, rpos = generate_beta_example_data(r_h, a_h, b_h, max_val = objective_range,nobj = n_obj, n_items=n_items, seed=items_seed)
    itemsdf = pd.DataFrame(items, columns= list(hard_items.columns[1:])+['Cost'])
    #this creates a mapping between the template and the trial data 
    items_sorted_hard_data, perm = sort_B_to_match_A_ranks(hard_data, items[:, :n_obj])
    items_sorted_df = itemsdf.iloc[perm].reset_index(drop=True)
    items_sorted_df.insert(0, hard_items.columns[0], hard_items.iloc[:,0].values)
    #sns.histplot(itemsdf,multiple = 'dodge',kde=True,bins = np.arange(0.5, objective_range + 0.5, 1))
    for k in range(n_obj):
        corrs_h[j,k] = spearmanr(items_sorted_hard_data[:, k], hard_data[:, k]).correlation
    items_sorted_df.to_csv(f'stimuli/civ_items_{items_seed}_trial_{itrial}.csv', index=False)
    itrial += 1
n_obj = 5
corrs_5 = np.zeros((nfive,n_obj))
for j in range(nfive):
    items_seed = subject_seed + nprac + neasy + nhard + j
    items, rpos = generate_beta_example_data(r, a, b, max_val = objective_range,nobj = n_obj, n_items=n_items, seed=items_seed)
    itemsdf = pd.DataFrame(items, columns= list(city_items.columns[1:])+['Cost'])
    #this creates a mapping between the template and the trial data 
    items_sorted_citydata, perm = sort_B_to_match_A_ranks(city_data, items[:, :n_obj])
    items_sorted_df = itemsdf.iloc[perm].reset_index(drop=True)
    items_sorted_df.insert(0, city_items.columns[0], city_items.iloc[:,0].values)
    #sns.histplot(itemsdf,multiple = 'dodge',kde=True,bins = np.arange(0.5, objective_range + 0.5, 1))
    for k in range(n_obj):
        corrs_5[j,k] = spearmanr(items_sorted_citydata[:, k], city_data[:, k]).correlation
    items_sorted_df.to_csv(f'stimuli/civ_items_{items_seed}_trial_{j}.csv', index=False)


In [75]:
n_selected = 6
#capacity = int(shape[-1]*scale[-1]*n_selected)
capacity = int(a[-1]/(a[-1]+b[-1])*objective_range*n_selected)
#print(capacity)
n_selected = 10
#capacity = int(shape[-1]*scale[-1]*n_selected)
capacity = int(a[-1]/(a[-1]+b[-1])*objective_range*n_selected)
#print(capacity)

In [76]:
if any(corrs_p.flatten() < 0.75):
    print("Warning: some practice trials have low correlation with template.")
if any(corrs_e.flatten() < 0.75):
    print("Warning: some easy trials have low correlation with template.")
if any(corrs_h.flatten() < 0.75):
    print("Warning: some hard trials have low correlation with template.")
if any(corrs_5.flatten() < 0.75):
    print("Warning: some five-objective trials have low correlation with template.")